# CLEF CheckThat! 2026 Task 2 — Single Experiment: NoAttn + Dup_1024

Replicates one config from the original ablation notebook:
- **use_pool = False** (no attention pooling over traces)
- **use_disagreement = True** (masked_std kept)
- **use_claim = True** (always kept)
- **verdict features expanded 7 -> 1024 via Cyclic Duplication** (tile+slice, no training)

Input dim = claim(4096) + disagreement(4096) + verdict(1024) = **9216**

Outputs: confusion matrix, per-class precision/recall/F1, macro-F1, and softmax probabilities
for False/True/Conflicting + predicted verdict, on train/val/test.

## 1. Environment Setup

In [1]:
!pip install "transformers>=4.40.0" accelerate bitsandbytes tqdm -q
print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 48.0 MB/s eta 0:00:00:00:0100:01
Installation complete.


In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

import warnings, os, gc, time, random, csv, copy
warnings.filterwarnings("ignore")

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU {_i}: {_p.name}  {_p.total_memory/1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


## 2. Paths & Config

In [3]:
# ============================================================
# PATHS  <- adjust to where you upload the input files
# ============================================================
DATA_DIR         = "/kaggle/input/clef-2026-checkthat-task-2-data"   # <- change if needed
TRAIN_JSON       = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/train.json"
VAL_JSON         = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/validation.json"
TEST_JSON        = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/test.json"
SEMANTIC_MATCHES = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/semantic_matches.json"

OUTPUT_DIR     = "/kaggle/working"
TRAIN_EMB_PATH = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/train_qwen3_embeddings.npz"
VAL_EMB_PATH   = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/val_qwen3_embeddings.npz"
TEST_EMB_PATH  = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/test_qwen3_embeddings.npz"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# EMBEDDING MODEL CONFIG (verified from Qwen/Qwen3-Embedding-8B card)
# ============================================================
EMBEDDING_MODEL_ID   = "Qwen/Qwen3-Embedding-8B"
EMBEDDING_BATCH_SIZE = 4
MAX_SEQ_LENGTH       = 8192
NORMALIZE_EMBEDDINGS = True

# ============================================================
# DATA FIELD NAMES
# ============================================================
CLAIM_FIELD        = "claim"
TRACE_TEXT_FIELD   = "reasoning_traces"
VERDICT_LIST_FIELD = "Verdict_list"

# ============================================================
# LABEL MAPPING
# ============================================================
label_map         = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {0: "False", 1: "True", 2: "Conflicting"}

# ============================================================
# MODEL / TRAINING CONFIG
# ============================================================
BATCH_SIZE          = 16
MAX_ITERS           = 100
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 20
WEIGHT_DECAY        = 1e-5
GRAD_CLIP_NORM      = 1.0
TRACE_MAX           = 20
SCALAR_DIM          = 7
ATTN_HIDDEN         = 256
HIDDEN_DIMS         = [512, 256, 128]
OUTPUT_DIM          = 3
DEVICE              = "cuda" if torch.cuda.is_available() else "cpu"

EMB_DIM              = 4096
EXPANDED_SCALAR_DIM  = 1024   # 7 -> 1024 via Cyclic Duplication

# THIS EXPERIMENT'S CONFIG: NoAttn + Dup_1024
EXP_NAME   = "C_NoAttn_Dup_1024"
EXP_CONFIG = dict(use_pool=False, use_disagreement=True)   # claim + verdict always kept

print("=" * 62)
print(f"Experiment    : {EXP_NAME}")
print(f"Config        : {EXP_CONFIG}")
print(f"Device        : {DEVICE}  ({torch.cuda.device_count()} GPU(s))")
print(f"EMB_DIM       : {EMB_DIM}")
print("=" * 62)

Experiment    : C_NoAttn_Dup_1024
Config        : {'use_pool': False, 'use_disagreement': True}
Device        : cuda  (2 GPU(s))
EMB_DIM       : 4096


## 3. Reproducibility

In [4]:
RANDOM_SEED   = 42
DETERMINISTIC = True

def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

set_seed()
print(f"Seed {RANDOM_SEED} set | deterministic={DETERMINISTIC}")

Seed 42 set | deterministic=True


## 4. Dataset Loading

In [5]:
def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def numpy_to_python(obj):
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, dict):          return {k: numpy_to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [numpy_to_python(i) for i in obj]
    return obj

print("Loading JSON datasets...")
_train_data       = load_json(TRAIN_JSON)
_val_data         = load_json(VAL_JSON)
_test_data        = load_json(TEST_JSON)
_semantic_matches = load_json(SEMANTIC_MATCHES)

print(f"Train items       : {len(_train_data)}")
print(f"Val items         : {len(_val_data)}")
print(f"Test items        : {len(_test_data)}")
print(f"Semantic matches  : {len(_semantic_matches)}")

Loading JSON datasets...
Train items       : 6400
Val items         : 1600
Test items        : 2558
Semantic matches  : 6469


In [6]:
def create_train_id_to_item() -> dict:
    claim_to_item  = {item[CLAIM_FIELD]: item for item in _train_data}
    new_id_to_item = {}
    for match in _semantic_matches:
        new_claim = match["new_claim"]
        if new_claim in claim_to_item:
            new_id_to_item[match["new_id"]] = claim_to_item[new_claim]
    print(f"Train ID->item  : {len(new_id_to_item)} entries mapped")
    return new_id_to_item

def create_val_id_to_item() -> dict:
    mapping = {idx: item for idx, item in enumerate(_val_data)}
    print(f"Val ID->item    : {len(mapping)} entries")
    return mapping

def create_test_id_to_item() -> dict:
    mapping = {item["query_id"]: item for item in _test_data}
    print(f"Test ID->item   : {len(mapping)} entries")
    return mapping

train_id_to_item = create_train_id_to_item()
val_id_to_item   = create_val_id_to_item()
test_id_to_item  = create_test_id_to_item()

Train ID->item  : 6400 entries mapped
Val ID->item    : 1600 entries
Test ID->item   : 2558 entries


## 5. Qwen3-Embedding-8B Setup

In [7]:
print(f"Loading tokeniser : {EMBEDDING_MODEL_ID}")
_emb_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_ID, trust_remote_code=True)

print(f"Loading model     : {EMBEDDING_MODEL_ID}  (torch.float16, device_map=auto)")
_emb_model = AutoModel.from_pretrained(
    EMBEDDING_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
_emb_model.eval()

_actual_emb_dim = _emb_model.config.hidden_size
print(f"\nmodel.config.hidden_size : {_actual_emb_dim}")
if _actual_emb_dim != EMB_DIM:
    print(f"  EMB_DIM was {EMB_DIM}; updating to {_actual_emb_dim}")
    EMB_DIM = _actual_emb_dim
else:
    print(f"  EMB_DIM = {EMB_DIM} confirmed")

Loading tokeniser : Qwen/Qwen3-Embedding-8B


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model     : Qwen/Qwen3-Embedding-8B  (torch.float16, device_map=auto)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


model.config.hidden_size : 4096
  EMB_DIM = 4096 confirmed


In [8]:
def last_token_pool(last_hidden_states: torch.Tensor,
                    attention_mask:     torch.Tensor) -> torch.Tensor:
    """Last non-padding token pool -- official Qwen3-Embedding strategy."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size       = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths,
    ]

def _get_trace_text(trace) -> str:
    if isinstance(trace, str):
        return trace
    if isinstance(trace, dict):
        for key in ("text", "content", "response", "reasoning", "trace", "passage"):
            if key in trace:
                return str(trace[key])
        return str(trace)
    return str(trace)

def get_item_texts(item: dict):
    claim_text  = item.get(CLAIM_FIELD, item.get("claim_text", "")).strip()
    traces_raw  = item.get(TRACE_TEXT_FIELD, [])
    trace_texts = [_get_trace_text(t) for t in traces_raw]
    return claim_text, trace_texts

@torch.no_grad()
def encode_texts(texts: list,
                 batch_size: int  = EMBEDDING_BATCH_SIZE,
                 max_length: int  = MAX_SEQ_LENGTH,
                 normalize:  bool = NORMALIZE_EMBEDDINGS) -> np.ndarray:
    """Encode texts -> (N, EMB_DIM) float32 using Qwen3-Embedding-8B."""
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch   = texts[start: start + batch_size]
        encoded = _emb_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        first_device = next(_emb_model.parameters()).device
        encoded = {k: v.to(first_device) for k, v in encoded.items()}
        outputs    = _emb_model(**encoded)
        embeddings = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
        if normalize:
            embeddings = F.normalize(embeddings.float(), p=2, dim=-1)
        else:
            embeddings = embeddings.float()
        all_embeddings.append(embeddings.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

## 6. Generate / Load Cached Embeddings

In [9]:
def generate_embeddings_for_split(id_to_item, id_order, split_name, cache_path):
    """
    Row 0   = claim embedding  (EMB_DIM,)
    Rows 1+ = trace embeddings (num_traces, EMB_DIM)
    Stored as plain dict (NOT NpzFile) -- required for DataLoader thread safety.
    """
    if os.path.exists(cache_path):
        print(f"[{split_name}] Cache found -- loading from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=False))

    print(f"\n[{split_name}] Generating embeddings for {len(id_order)} items ...")
    emb_dict = {}

    for id_ in tqdm(id_order, desc=f"[{split_name}]"):
        item = id_to_item[id_]
        claim_text, trace_texts = get_item_texts(item)
        texts = [claim_text] + trace_texts
        arr   = encode_texts(texts)
        emb_dict[f"id_{id_}"] = arr

    np.savez_compressed(cache_path, **emb_dict)
    print(f"[{split_name}] Saved -> {cache_path}")
    return emb_dict

train_emb_data = generate_embeddings_for_split(
    train_id_to_item, sorted(train_id_to_item.keys()), "TRAIN", TRAIN_EMB_PATH)
val_emb_data = generate_embeddings_for_split(
    val_id_to_item, sorted(val_id_to_item.keys()), "VAL", VAL_EMB_PATH)
test_emb_data = generate_embeddings_for_split(
    test_id_to_item, sorted(test_id_to_item.keys()), "TEST", TEST_EMB_PATH)

[TRAIN] Cache found -- loading from /kaggle/input/datasets/d202511054/qwen3-embeddings/train_qwen3_embeddings.npz
[VAL] Cache found -- loading from /kaggle/input/datasets/d202511054/qwen3-embeddings/val_qwen3_embeddings.npz
[TEST] Cache found -- loading from /kaggle/input/datasets/d202511054/qwen3-embeddings/test_qwen3_embeddings.npz


## 7. Feature Engineering

In [10]:
def split_and_pad(emb_array: np.ndarray, max_traces=None, emb_dim=None):
    """Row 0=claim, Rows 1+=traces. Zero-pad traces to TRACE_MAX."""
    if max_traces is None: max_traces = TRACE_MAX
    if emb_dim    is None: emb_dim    = EMB_DIM

    claim_emb = emb_array[0].astype(np.float32)
    traces    = emb_array[1:].astype(np.float32)
    n         = traces.shape[0]
    mask      = np.zeros(max_traces, dtype=np.float32)

    if n < max_traces:
        pad           = np.zeros((max_traces - n, emb_dim), dtype=np.float32)
        traces_padded = np.vstack([traces, pad])
        mask[:n]      = 1.0
    else:
        traces_padded     = traces[:max_traces]
        mask[:max_traces] = 1.0

    return claim_emb, traces_padded, mask


def verdict_features(verdict_list: list, max_traces=None) -> np.ndarray:
    """7 scalar features from per-trace vote distribution."""
    if max_traces is None: max_traces = TRACE_MAX

    counts = {0: 0, 1: 0, 2: 0}
    for v in verdict_list:
        if v in label_map:
            counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0:
        return np.zeros(SCALAR_DIM, dtype=np.float32)

    frac_false      = counts[0] / total
    frac_true       = counts[1] / total
    frac_conf       = counts[2] / total
    majority_frac   = max(counts.values()) / total
    distinct_ratio  = sum(1 for c in counts.values() if c > 0) / 3.0
    fracs           = [frac_false, frac_true, frac_conf]
    entropy         = -sum(p * np.log(p + 1e-9) for p in fracs if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)

    return np.array(
        [frac_false, frac_true, frac_conf,
         majority_frac, distinct_ratio, entropy, num_traces_norm],
        dtype=np.float32,
    )

print(f"split_and_pad    -> claim=({EMB_DIM},) traces=({TRACE_MAX},{EMB_DIM}) mask=({TRACE_MAX},)")
print(f"verdict_features -> ({SCALAR_DIM},)")

split_and_pad    -> claim=(4096,) traces=(20,4096) mask=(20,)
verdict_features -> (7,)


## 8. Model Architecture: Disagreement + Cyclic Duplication (No Attention Pooling)

In [11]:
# ── Disagreement Vector (masked std across trace embeddings) ─────────────────
def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)
    count  = mask_f.sum(dim=1).clamp(min=1.0)
    mean   = (traces * mask_f).sum(dim=1) / count
    var    = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)


# ── Verdict Expansion: Cyclic Duplication (7 -> 1024, no training) ───────────
class CyclicDuplication(nn.Module):
    """Repeat verdict vector ceil(1024/7)=147 times then slice to 1024."""
    def __init__(self, in_dim: int = SCALAR_DIM, out_dim: int = EXPANDED_SCALAR_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.repeats = (out_dim + in_dim - 1) // in_dim   # = 147
        print(f"  CyclicDuplication: {in_dim} x {self.repeats} = "
              f"{in_dim*self.repeats}  ->  slice to {out_dim}")
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.repeat(1, self.repeats)[:, :self.out_dim]   # (B, 1024)


# ── Configurable Expanded Model (NoAttn variant: use_pool=False) ─────────────
class ConfigurableExpandedTraceMLP(nn.Module):
    """
    claim + [attn_pool if use_pool] + [masked_std if use_disagreement] + verdict_expander(scalars)
    -> BatchNorm -> [Linear->BN->ReLU->Dropout]x3 -> Linear(3)

    For this experiment: use_pool=False, use_disagreement=True, verdict_expander=CyclicDuplication
    """
    def __init__(self, exp_config: dict, verdict_expander: nn.Module,
                 emb_dim=None, expanded_scalar_dim=EXPANDED_SCALAR_DIM,
                 hidden_dims=None, output_dim=OUTPUT_DIM):
        super().__init__()
        if emb_dim is None: emb_dim = EMB_DIM
        if hidden_dims is None: hidden_dims = HIDDEN_DIMS

        self.use_pool  = exp_config.get("use_pool", True)
        self.use_disag = exp_config.get("use_disagreement", True)
        self.verdict_expander = verdict_expander

        if self.use_pool:
            # kept for architecture generality; unused when use_pool=False
            self.attn_pool = nn.Sequential(
                nn.Linear(emb_dim, ATTN_HIDDEN), nn.Tanh(), nn.Linear(ATTN_HIDDEN, 1))

        combined_dim = emb_dim  # claim always kept
        if self.use_pool:  combined_dim += emb_dim
        if self.use_disag: combined_dim += emb_dim
        combined_dim += expanded_scalar_dim  # verdict always kept
        self.combined_dim = combined_dim

        layers = [nn.BatchNorm1d(combined_dim)]
        prev = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        parts = [claim]
        if self.use_pool:
            scores  = self.attn_pool(traces).squeeze(-1)
            scores  = scores.masked_fill(mask == 0, float("-inf"))
            weights = torch.softmax(scores, dim=1)
            parts.append(torch.bmm(weights.unsqueeze(1), traces).squeeze(1))
        if self.use_disag:
            parts.append(masked_std(traces, mask))
        parts.append(self.verdict_expander(scalars))
        return self.mlp(torch.cat(parts, dim=1))


# ── Shape check ────────────────────────────────────────────────────────────
_c, _tr, _mk, _sc = (torch.randn(2, EMB_DIM).to(DEVICE), torch.randn(2, TRACE_MAX, EMB_DIM).to(DEVICE),
                      torch.ones(2, TRACE_MAX).to(DEVICE), torch.randn(2, SCALAR_DIM).to(DEVICE))
_m = ConfigurableExpandedTraceMLP(EXP_CONFIG, CyclicDuplication()).to(DEVICE)
with torch.no_grad():
    _o = _m(_c, _tr, _mk, _sc)
print(f"combined_dim={_m.combined_dim}  out={tuple(_o.shape)}")
assert _o.shape == (2, OUTPUT_DIM)
del _c, _tr, _mk, _sc, _m, _o

  CyclicDuplication: 7 x 147 = 1029  ->  slice to 1024
combined_dim=9216  out=(2, 3)


## 9. PyTorch Datasets

In [12]:
class BaseDataset(Dataset):
    label_key = "label"
    def __getitem__(self, idx):
        id_       = self.ids[idx]
        emb_array = self.emb_data[f"id_{id_}"]
        claim_emb, traces_padded, mask = split_and_pad(emb_array)
        item         = self.id_to_item[id_]
        label        = label_map[item[self.label_key]]
        verdict_list = item.get(VERDICT_LIST_FIELD, [])
        scalars      = verdict_features(verdict_list)
        return (
            torch.from_numpy(claim_emb),
            torch.from_numpy(traces_padded),
            torch.from_numpy(mask),
            torch.from_numpy(scalars),
            torch.tensor(label, dtype=torch.long),
        )
    def __len__(self): return len(self.ids)

class TrainDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data   = train_emb_data
        self.id_to_item = train_id_to_item
        all_ids  = sorted(int(k.split("_")[1]) for k in self.emb_data.keys() if k.startswith("id_"))
        self.ids = [i for i in all_ids if i in self.id_to_item]
        print(f"TRAIN dataset : {len(self.ids)} samples")

class ValDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data   = val_emb_data
        self.id_to_item = val_id_to_item
        self.ids        = sorted(self.id_to_item.keys())
        print(f"VAL   dataset : {len(self.ids)} samples")

class TestDataset(BaseDataset):
    label_key = "Label"   # capital L in test set
    def __init__(self):
        self.emb_data   = test_emb_data
        self.id_to_item = test_id_to_item
        self.ids        = sorted(self.id_to_item.keys())
        print(f"TEST  dataset : {len(self.ids)} samples")

set_seed()
train_dataset = TrainDataset()
val_dataset   = ValDataset()
test_dataset  = TestDataset()

_train_label_list = [label_map[train_dataset.id_to_item[i]["label"]] for i in train_dataset.ids]
_label_counts = Counter(_train_label_list)
_total        = sum(_label_counts.values())
class_weights = torch.tensor(
    [_total / (3.0 * _label_counts[c]) for c in range(3)], dtype=torch.float32
).to(DEVICE)

print(f"\nLabel distribution : {dict(_label_counts)}")
print(f"Class weights      : {[round(w, 4) for w in class_weights.tolist()]}")

# num_workers=0 -- NpzFile/dict in forked workers causes zlib errors on Kaggle
_dl_kw = dict(num_workers=0, pin_memory=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_dl_kw)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

TRAIN dataset : 6400 samples
VAL   dataset : 1600 samples
TEST  dataset : 2558 samples

Label distribution : {2: 1508, 0: 3717, 1: 1175}
Class weights      : [0.5739, 1.8156, 1.4147]

Train batches : 400
Val batches   : 100
Test batches  : 160


## 10. Training / Evaluation Pipeline

In [13]:
def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0.0
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True),   traces.to(device, non_blocking=True),
            mask.to(device,  non_blocking=True),   scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                logits = model(claim, traces, mask, scalars)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(claim, traces, mask, scalars)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_probs=False):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True),   traces.to(device, non_blocking=True),
            mask.to(device,  non_blocking=True),   scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True),
        )
        logits = model(claim, traces, mask, scalars)
        total_loss += criterion(logits, labels).item()
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        if return_probs:
            all_probs.extend(probs.cpu().numpy().tolist())
    acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    if return_probs:
        return total_loss / len(loader), acc, all_preds, all_labels, all_probs
    return total_loss / len(loader), acc, all_preds, all_labels


def generate_report(all_labels, all_preds, set_name="VAL", save_prefix=None):
    all_labels = np.array(all_labels, dtype=int)
    all_preds  = np.array(all_preds,  dtype=int)
    cm  = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2])
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n{'='*70}\n{set_name} SET -- METRICS\n{'='*70}")
    print("Confusion Matrix (rows=true, cols=pred | False/True/Conflicting):")
    print(cm, "\n")

    precision = precision_score(all_labels, all_preds, average=None, zero_division=0, labels=[0,1,2])
    recall    = recall_score(   all_labels, all_preds, average=None, zero_division=0, labels=[0,1,2])
    f1        = f1_score(       all_labels, all_preds, average=None, zero_division=0, labels=[0,1,2])

    print(f"{'Class':<10}{'Label':<16}{'Precision':<12}{'Recall':<12}{'F1':<12}")
    print("-"*62)
    for c in range(3):
        print(f"{c:<10}{reverse_label_map[c]:<16}{precision[c]:<12.4f}{recall[c]:<12.4f}{f1[c]:<12.4f}")
    print("-"*62)
    macro_p, macro_r, macro_f1 = float(np.mean(precision)), float(np.mean(recall)), float(np.mean(f1))
    print(f"{'MACRO AVG':<26}{macro_p:<12.4f}{macro_r:<12.4f}{macro_f1:<12.4f}\n")

    wp  = float(precision_score(all_labels, all_preds, average="weighted", zero_division=0))
    wr  = float(recall_score(   all_labels, all_preds, average="weighted", zero_division=0))
    wf1 = float(f1_score(       all_labels, all_preds, average="weighted", zero_division=0))
    print(f"Accuracy: {acc:.4f}  |  Weighted F1: {wf1:.4f}")

    pfx = save_prefix if save_prefix else set_name.lower().replace(" ", "_")
    try:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=list(reverse_label_map.values()),
                    yticklabels=list(reverse_label_map.values()))
        ax.set_title(f"{set_name} -- Confusion Matrix")
        ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{pfx}_confusion_matrix.png", dpi=100, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        print(f"Warning: plot failed: {exc}")

    return {
        "accuracy": float(acc), "cm": cm.tolist(),
        "precision": numpy_to_python(precision), "recall": numpy_to_python(recall),
        "f1": numpy_to_python(f1),
        "avg_precision": macro_p, "avg_recall": macro_r, "avg_f1": macro_f1,
        "macro_f1": macro_f1,
        "weighted_precision": wp, "weighted_recall": wr, "weighted_f1": wf1,
    }

## 11. Run Experiment: NoAttn + Dup_1024

In [14]:
def run_configurable_expanded_experiment(exp_name, exp_config, verdict_expander,
                                         train_loader, val_loader, test_loader,
                                         weights, device=DEVICE, use_amp=True):
    print(f"\n{'#'*72}\n# EXPERIMENT : {exp_name}\n# Config: {exp_config}\n{'#'*72}")
    set_seed()
    verdict_expander = verdict_expander.to(device)
    model = ConfigurableExpandedTraceMLP(exp_config, verdict_expander).to(device)
    _cdim = model.combined_dim
    print(f"combined_dim={_cdim}  params={sum(p.numel() for p in model.parameters()):,}")

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"DataParallel across {torch.cuda.device_count()} GPUs")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = GradScaler() if (use_amp and device.startswith("cuda")) else None

    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    train_losses, val_losses = [], []
    t0 = time.time()

    for epoch in range(MAX_ITERS):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr_loss); val_losses.append(va_loss)
        scheduler.step(va_loss)
        print(f"  Ep {epoch+1:3d}/{MAX_ITERS} | Train {tr_loss:.4f} | Val {va_loss:.4f} | ValAcc {va_acc:.4f}")
        if va_loss < best_val_loss:
            best_val_loss, patience_counter = va_loss, 0
            _base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = copy.deepcopy(_base.state_dict())
            print("    Best model saved")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - t0
    _base = model.module if isinstance(model, nn.DataParallel) else model
    _base.load_state_dict(best_state)

    pfx = exp_name.lower().replace(" ", "_")
    tr_f, _, tr_p, tr_l, tr_probs = evaluate(model, train_loader, criterion, device, return_probs=True)
    va_f, _, va_p, va_l, va_probs = evaluate(model, val_loader,   criterion, device, return_probs=True)
    te_f, _, te_p, te_l, te_probs = evaluate(model, test_loader,  criterion, device, return_probs=True)

    tr_m = generate_report(tr_l, tr_p, f"{exp_name} TRAIN", f"{pfx}_train")
    va_m = generate_report(va_l, va_p, f"{exp_name} VAL",   f"{pfx}_val")
    te_m = generate_report(te_l, te_p, f"{exp_name} TEST",  f"{pfx}_test")

    result = {
        "experiment": exp_name,
        "config": {**exp_config, "expanded_scalar_dim": EXPANDED_SCALAR_DIM, "combined_dim": _cdim},
        "train_loss": float(tr_f), "val_loss": float(va_f), "test_loss": float(te_f),
        "train_metrics": tr_m, "val_metrics": va_m, "test_metrics": te_m,
        "train_losses": [float(x) for x in train_losses],
        "val_losses": [float(x) for x in val_losses],
        "training_time_s": float(elapsed),
    }
    probs_data = {
        "train": {"ids": train_dataset.ids, "labels": tr_l, "preds": tr_p, "probs": tr_probs},
        "val":   {"ids": val_dataset.ids,   "labels": va_l, "preds": va_p, "probs": va_probs},
        "test":  {"ids": test_dataset.ids,  "labels": te_l, "preds": te_p, "probs": te_probs},
    }
    del model; gc.collect(); torch.cuda.empty_cache()
    return result, probs_data

print("run_configurable_expanded_experiment() ready.")

run_configurable_expanded_experiment() ready.


In [15]:
result, probs_data = run_configurable_expanded_experiment(
    exp_name=EXP_NAME,
    exp_config=EXP_CONFIG,
    verdict_expander=CyclicDuplication(),
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    weights=class_weights,
)

print("\n" + "="*70)
print(f"FINAL SUMMARY -- {EXP_NAME}")
print("="*70)
print(f"Train Macro F1 : {result['train_metrics']['macro_f1']:.4f}")
print(f"Val   Macro F1 : {result['val_metrics']['macro_f1']:.4f}")
print(f"Test  Macro F1 : {result['test_metrics']['macro_f1']:.4f}")
print("="*70)

  CyclicDuplication: 7 x 147 = 1029  ->  slice to 1024

########################################################################
# EXPERIMENT : C_NoAttn_Dup_1024
# Config: {'use_pool': False, 'use_disagreement': True}
########################################################################
combined_dim=9216  params=4,903,939
DataParallel across 2 GPUs
  Ep   1/100 | Train 0.9842 | Val 1.2848 | ValAcc 0.5706
    Best model saved
  Ep   2/100 | Train 0.9212 | Val 2.1481 | ValAcc 0.5706
  Ep   3/100 | Train 0.8695 | Val 1.8289 | ValAcc 0.5706
  Ep   4/100 | Train 0.8225 | Val 1.4930 | ValAcc 0.5706
  Ep   5/100 | Train 0.8073 | Val 2.6979 | ValAcc 0.5706
  Ep   6/100 | Train 0.6955 | Val 2.8625 | ValAcc 0.5706
  Ep   7/100 | Train 0.6469 | Val 1.8353 | ValAcc 0.5744
  Ep   8/100 | Train 0.5903 | Val 4.2359 | ValAcc 0.5706
  Ep   9/100 | Train 0.5552 | Val 2.8610 | ValAcc 0.5713
  Ep  10/100 | Train 0.4661 | Val 2.2984 | ValAcc 0.5775
  Ep  11/100 | Train 0.4473 | Val 4.1515 | ValAcc 0.570

## 12. Per-Sample Prediction Probabilities (False / True / Conflicting) + Verdict

In [16]:
import pandas as pd

def build_probs_dataframe(split_probs, id_to_item, split_name):
    rows = []
    for id_, true_label, pred_label, prob in zip(
            split_probs["ids"], split_probs["labels"], split_probs["preds"], split_probs["probs"]):
        rows.append({
            "id": id_,
            "true_label": reverse_label_map[true_label],
            "predicted_verdict": reverse_label_map[pred_label],
            "prob_False": prob[0],
            "prob_True": prob[1],
            "prob_Conflicting": prob[2],
        })
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUTPUT_DIR}/{EXP_NAME.lower()}_{split_name}_probabilities.csv", index=False)
    return df

train_probs_df = build_probs_dataframe(probs_data["train"], train_id_to_item, "train")
val_probs_df   = build_probs_dataframe(probs_data["val"],   val_id_to_item,   "val")
test_probs_df  = build_probs_dataframe(probs_data["test"],  test_id_to_item,  "test")

print("TEST set -- first 15 rows of probabilities + predicted verdict:")
test_probs_df.head(15)

TEST set -- first 15 rows of probabilities + predicted verdict:


,id,true_label,predicted_verdict,prob_False,prob_True,prob_Conflicting
0,0,False,False,0.749331,0.076215,0.174454
1,1,False,False,0.767445,0.025502,0.207053
2,2,False,False,0.662661,0.107759,0.229580
3,3,False,False,0.742728,0.039080,0.218192
4,4,False,Conflicting,0.255331,0.340781,0.403888
5,5,False,Conflicting,0.292025,0.351935,0.356040
6,6,False,False,0.837377,0.023277,0.139345
7,7,False,False,0.487919,0.296819,0.215262
8,8,False,False,0.616182,0.156508,0.227311
9,9,False,False,0.717148,0.059847,0.223005


## 13. Save Results

In [17]:
with open(f"{OUTPUT_DIR}/{EXP_NAME.lower()}_results.json", "w") as f:
    json.dump(numpy_to_python(result), f, indent=2)

print(f"Results saved -> {OUTPUT_DIR}/{EXP_NAME.lower()}_results.json")
print(f"Probabilities saved -> {OUTPUT_DIR}/{EXP_NAME.lower()}_{{train,val,test}}_probabilities.csv")
print(f"Confusion matrix plots -> {OUTPUT_DIR}/{EXP_NAME.lower()}_{{train,val,test}}_confusion_matrix.png")

Results saved -> /kaggle/working/c_noattn_dup_1024_results.json
Probabilities saved -> /kaggle/working/c_noattn_dup_1024_{train,val,test}_probabilities.csv
Confusion matrix plots -> /kaggle/working/c_noattn_dup_1024_{train,val,test}_confusion_matrix.png
